In [1]:
import os
import time
import glob

import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import MinMaxScaler

from qiskit.circuit.library import PauliFeatureMap, RealAmplitudes, ZZFeatureMap
from qiskit_machine_learning.algorithms import VQR
from qiskit_algorithms.optimizers import L_BFGS_B

from joblib import Parallel, delayed

try:
    from tqdm import tqdm
except ImportError:  # tqdm is optional -- falls back to plain prints
    def tqdm(iterable, **kwargs):
        return iterable

In [2]:
root_folder = 'VQR'
np.random.seed(42)

NUM_FEATURES = 3

IF_PAULI_FEATURE_MAP_LIST = [True, False]
FEATURE_MAP_REPS_LIST = [1,2,3,4,5]
ANSATZ_REPS_LIST = [1,2,3,4,5]
ENTANGLEMENT_LIST = ['linear', 'full', 'circular']

LOSS_FUNCTION = 'squared_error'

N_REPEATS = 3
TEST_SIZE = 1

date = '05_31_25_0'
dataset_name = "/home/ashok/Desktop/ABHI/Learning,Reproducing/qml_training-validation-data.csv"

# How many CPU cores to use.
N_JOBS = 8

# Where per-fold checkpoints and the final merged CSV go
RESULT_DIR = f'{root_folder}/result'
FOLD_DIR = f'{RESULT_DIR}/folds'
LOG_DIR = f'{root_folder}/logs'

for d in (RESULT_DIR, FOLD_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

In [3]:
def prepare_dataset_k_fold(X, y, train_indices, test_indices):
    X_train_raw, X_test_raw = X[train_indices], X[test_indices]
    y_train, y_test = y[train_indices], y[test_indices]

    element_test = X_test_raw[:, 0]
    element_train = X_train_raw[:, 0]

    X_train = X_train_raw[:, 1:]
    X_test = X_test_raw[:, 1:]

    full_X = np.vstack([X_train, X_test])

    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(full_X)

    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, y_train, X_test_scaled, y_test, element_test, element_train


In [4]:
def reconfig_quantum_kernel_vqr(if_pauli_feature_map, feature_reps, ansatz_reps, entangle, objective_func_vals):
    def callback_graph(weights, obj_func_eval):
        objective_func_vals.append(obj_func_eval)

    if if_pauli_feature_map:
        feature_map = PauliFeatureMap(feature_dimension=NUM_FEATURES, reps=feature_reps, entanglement=entangle)
    else:
        feature_map = ZZFeatureMap(feature_dimension=NUM_FEATURES, reps=feature_reps, entanglement=entangle)

    ansatz = RealAmplitudes(num_qubits=NUM_FEATURES, reps=ansatz_reps)
    optimizer = L_BFGS_B(ftol=0.000001)

    return VQR(feature_map=feature_map,
               ansatz=ansatz,
               optimizer=optimizer,
               callback=callback_graph,
               loss=LOSS_FUNCTION,
               )

In [5]:
def train_vqr(vqr, X_train, y_train, X_test):
    vqr.fit(X_train, np.concatenate(y_train))
    return vqr.predict(X_train), vqr.predict(X_test)


In [6]:
def process_fold(fold_id, train_indices, test_indices, X, y, y_scaler):
    """
    Runs the full hyperparameter sweep for a single fold and writes a
    checkpoint CSV for that fold. Returns the checkpoint file path.

    If the checkpoint already exists (from a previous run), the fold is
    skipped entirely -- this is what gives us resume support.
    """
    checkpoint_path = f'{FOLD_DIR}/fold_{fold_id:04d}.csv'
    if os.path.exists(checkpoint_path):
        return checkpoint_path  # already done -- resume support

    X_train, y_train, X_test, y_test, element_test, element_train = \
        prepare_dataset_k_fold(X, y, train_indices, test_indices)

    rows = []

    for if_pauli_feature_map in IF_PAULI_FEATURE_MAP_LIST:
        for feature_map_reps in FEATURE_MAP_REPS_LIST:
            for ansatz_reps in ANSATZ_REPS_LIST:
                for entanglement in ENTANGLEMENT_LIST:

                    feature_map_name = 'Pauli' if if_pauli_feature_map else 'ZZ'

                    objective_func_vals = []
                    vqr = reconfig_quantum_kernel_vqr(
                        if_pauli_feature_map=if_pauli_feature_map,
                        feature_reps=feature_map_reps,
                        ansatz_reps=ansatz_reps,
                        entangle=entanglement,
                        objective_func_vals=objective_func_vals,
                    )

                    predict_train, predict_test = train_vqr(vqr, X_train, y_train, X_test)

                    all_preds = y_scaler.inverse_transform(np.array(predict_test).reshape(-1, 1))
                    all_targets = y_scaler.inverse_transform(np.array(y_test).reshape(-1, 1))

                    all_preds_train = y_scaler.inverse_transform(np.array(predict_train).reshape(-1, 1))
                    all_targets_train = y_scaler.inverse_transform(np.array(y_train).reshape(-1, 1))

                    new_row = {
                        'fold_id': fold_id,
                        'feature_map_name': feature_map_name,
                        'feature_map_reps': feature_map_reps,
                        'ansatz_reps': ansatz_reps,
                        'entanglement': entanglement,
                        'element test': element_test,
                        'actual test': np.array(all_targets).flatten(),
                        'predicted test': np.array(all_preds).flatten(),
                        'element train': element_train,
                        'actual train': np.array(all_targets_train).flatten(),
                        'predicted train': np.array(all_preds_train).flatten(),
                        # 'R2 test': r2_score(y_test, predict_test),  # left commented, same as your notebook
                        'R2 train': r2_score(y_train, predict_train),
                    }
                    rows.append(new_row)

    fold_df = pd.DataFrame(rows)
    # write atomically (write to tmp, then rename) so a crash mid-write
    # can't leave a half-written checkpoint that looks "done"
    tmp_path = checkpoint_path + '.tmp'
    with np.printoptions(linewidth=10000):
        fold_df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, checkpoint_path)

    return checkpoint_path

In [ ]:
if __name__ == '__main__':

    # ---- output filename (same naming scheme as your original notebook) ----
    def _name(lst):
        return lst[0] if len(lst) == 1 else lst

    FEATURE_MAP_REPS_LIST_NAME = _name(FEATURE_MAP_REPS_LIST)
    ANSATZ_REPS_LIST_NAME = _name(ANSATZ_REPS_LIST)
    ENTANGLEMENT_LIST_NAME = _name(ENTANGLEMENT_LIST)
    IF_PAULI_FEATURE_MAP_LIST_NAME = str(_name(IF_PAULI_FEATURE_MAP_LIST)) \
        .replace('False', 'ZZ').replace('True', 'Pauli')

    file_name = (f'{RESULT_DIR}/FMR_{FEATURE_MAP_REPS_LIST_NAME}_AR_{ANSATZ_REPS_LIST_NAME}'
                 f'_E_{ENTANGLEMENT_LIST_NAME}_P_{IF_PAULI_FEATURE_MAP_LIST_NAME}_{date}.csv')
    print('Final results will be merged into:', file_name)

    # ---- load data ----
    df = pd.read_csv(dataset_name)
    X = df[['Element', 'el_neg', 'B/GPa', 'Volume/A^3']].values
    y = df['SFE/mJm^-3'].values
    print('Dataset shape:', df.shape)

    # ---- scale target (regression -- no classification threshold here) ----
    y_scaler = MinMaxScaler(feature_range=(-1, 1))
    y = y_scaler.fit_transform(y.reshape(-1, 1))

    # ---- build folds ----
    print('Total number of data:', X.shape[0])
    rkf = RepeatedKFold(n_splits=X.shape[0] // TEST_SIZE, n_repeats=N_REPEATS)
    print(rkf)

    folds = list(enumerate(rkf.split(X)))
    n_folds = len(folds)
    n_combos = (len(IF_PAULI_FEATURE_MAP_LIST) * len(FEATURE_MAP_REPS_LIST)
                * len(ANSATZ_REPS_LIST) * len(ENTANGLEMENT_LIST))
    print(f'{n_folds} folds x {n_combos} hyperparameter combos '
          f'= {n_folds * n_combos} total experiments')

    already_done = sorted(glob.glob(f'{FOLD_DIR}/fold_*.csv'))
    if already_done:
        print(f'Resuming: {len(already_done)}/{n_folds} folds already have checkpoints and will be skipped.')

    # ---- run folds in parallel, N_JOBS at a time, streaming results live ----
    start = time.time()
    n_done = len(already_done)
    live_header_written = os.path.exists(file_name)

    # If resuming, seed the live CSV with whatever's already on disk so it's
    # not empty/stale while we wait for the next fold to land.
    if already_done and not live_header_written:
        seed = pd.concat((pd.read_csv(f) for f in already_done), ignore_index=True)
        seed.to_csv(file_name, index=False)
        live_header_written = True

    print(f'\n--- Dispatching {n_folds - n_done} remaining folds across {N_JOBS} workers ---\n')

    # return_as="generator_unordered" yields each fold's result the moment
    # that worker finishes, instead of waiting for the whole batch.
    parallel = Parallel(n_jobs=N_JOBS, backend='loky', return_as='generator_unordered')
    jobs = (
        delayed(process_fold)(fold_id, train_idx, test_idx, X, y, y_scaler)
        for fold_id, (train_idx, test_idx) in folds
    )

    for checkpoint_path in parallel(jobs):
        n_done += 1
        fold_df = pd.read_csv(checkpoint_path)
        mean_r2 = fold_df['R2 train'].mean()
        elapsed = time.time() - start
        rate = (n_done - len(already_done)) / elapsed if elapsed > 0 else 0
        eta_min = ((n_folds - n_done) / rate / 60) if rate > 0 else float('inf')

        print(f'[{n_done}/{n_folds}] {os.path.basename(checkpoint_path)} done | '
              f'mean R2(train)={mean_r2:.3f} | elapsed={elapsed/60:.1f}m | '
              f'ETA~{eta_min:.1f}m', flush=True)

        # append this fold's rows to the live-growing final CSV right away,
        # so the file on disk always reflects everything completed so far
        fold_df.to_csv(file_name, mode='a', index=False, header=not live_header_written)
        live_header_written = True
A)


Does this exist?


ls VQR/result/folds | wc -l
    total_elapsed = time.time() - start
    print(f'\nAll folds complete in {total_elapsed/60:.1f} minutes.')
    print(f'Results (growing live throughout the run) are in: {file_name}')

    final_check = pd.read_csv(file_name)
    print(f'Final shape: {final_check.shape}')

Final results will be merged into: VQR/result/FMR_[1, 2, 3, 4, 5]_AR_[1, 2, 3, 4, 5]_E_['linear', 'full', 'circular']_P_[Pauli, ZZ]_05_31_25_0.csv
Dataset shape: (21, 5)
Total number of data: 21
RepeatedKFold(n_repeats=3, n_splits=21, random_state=None)
63 folds x 150 hyperparameter combos = 9450 total experiments

--- Dispatching 63 remaining folds across 8 workers ---

[1/63] fold_0006.csv done | mean R2(train)=0.713 | elapsed=243.4m | ETA~15088.1m
[2/63] fold_0003.csv done | mean R2(train)=0.721 | elapsed=243.8m | ETA~7436.9m
[3/63] fold_0001.csv done | mean R2(train)=0.712 | elapsed=249.9m | ETA~4998.7m
[4/63] fold_0005.csv done | mean R2(train)=0.707 | elapsed=250.7m | ETA~3697.6m
[5/63] fold_0000.csv done | mean R2(train)=0.718 | elapsed=254.4m | ETA~2951.4m
[6/63] fold_0004.csv done | mean R2(train)=0.730 | elapsed=261.8m | ETA~2487.0m
[7/63] fold_0007.csv done | mean R2(train)=0.731 | elapsed=267.5m | ETA~2139.9m
[8/63] fold_0002.csv done | mean R2(train)=0.711 | elapsed=268.4m

In [14]:
import glob
import pandas as pd

for f in sorted(glob.glob("VQR/result/*.csv")):
    df = pd.read_csv(f)
    print(f"{f} -> {df.shape}")

VQR/result/FMR_[1, 2, 3, 4, 5]_AR_[1, 2, 3, 4, 5]_E_['linear', 'full', 'circular']_P_[Pauli, ZZ]_05_31_25_0.csv -> (9450, 12)
